## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-18-19-52-40 +0000,chitri,Aldermen approve investigation into Mayor Bran...,https://www.chicagotribune.com/2026/11/18/alde...
1,2026-11-18-19-51-39 +0000,nyt,"What Nvidia’s Interim CEO, Colette Kress, Said...",https://www.nytimes.com/2026/11/18/technology/...
2,2026-11-18-19-51-08 +0000,nyt,"NYC, LA and Chicago School Districts Weigh Bre...",https://www.nytimes.com/2026/11/18/technology/...
3,2026-11-18-19-49-35 +0000,nypost,Russia gives stark response as France sends so...,https://nypost.com/2026/11/18/world-news/russi...
4,2026-11-18-19-48-27 +0000,nyt,Fed Saw Need to Continue Raising Rates Despite...,https://www.nytimes.com/2026/11/18/business/fe...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2754/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
43,trump,69
8,nvidia,34
81,new,25
15,huang,20
44,xi,19


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
378,2026-11-17-23-02-12 +0000,bbc,"As Trump and Xi meet, China's AI dreams face a...",https://www.bbc.co.uk/news/articles/c5vg23g92r...,178
40,2026-11-18-18-41-56 +0000,chitri,President Trump plans separate talks with Vlad...,https://www.chicagotribune.com/2026/11/18/trum...,168
37,2026-11-18-18-52-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...,167
36,2026-11-18-18-52-04 +0000,nyt,Trump Arrives in China Focused on Friendship W...,https://www.nytimes.com/2026/11/18/us/politics...,158
131,2026-11-18-16-10-44 +0000,bbc,Trump shakes hands with Putin and Xi ahead of ...,https://www.bbc.co.uk/news/articles/ck9d75x5ww...,146


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
378,178,2026-11-17-23-02-12 +0000,bbc,"As Trump and Xi meet, China's AI dreams face a...",https://www.bbc.co.uk/news/articles/c5vg23g92r...
301,105,2026-11-18-04-23-00 +0000,wsj,Democrat Mary Peltola unseated Republican Sen....,https://www.wsj.com/politics/elections/democra...
37,94,2026-11-18-18-52-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...
191,80,2026-11-18-12-56-00 +0000,wsj,The Morning Risk Report: SEC Is Probing Nvidia...,https://www.wsj.com/risk-compliance-journal/th...
393,61,2026-11-17-22-38-07 +0000,latimes,'What a mistake.' Popular Studio City restaura...,https://www.latimes.com/food/story/2026-11-17/...
397,56,2026-11-17-22-33-00 +0000,wsj,"Hung Cao, President Trump’s pick to lead the N...",https://www.wsj.com/politics/national-security...
201,55,2026-11-18-12-07-00 +0000,wsj,"In Choosing Shenzhen for Trump-Putin Summit, X...",https://www.wsj.com/world/china/in-choosing-sh...
433,54,2026-11-17-21-25-44 +0000,wapo,Jeffries seeks to shore up support ahead of vo...,https://www.washingtonpost.com/politics/2026/1...
207,53,2026-11-18-11-57-00 +0000,wsj,Homeland Security Secretary Markwayne Mullin h...,https://www.wsj.com/politics/policy/markwayne-...
317,44,2026-11-18-02-38-00 +0000,wsj,Some of President Trump’s political allies are...,https://www.wsj.com/politics/policy/cascade-of...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
